# Big Data Analytics (BDA) & Spatial Gap Analysis — Surabaya Timur
### Pemetaan Geospasial, Autokorelasi Spasial, dan Analisis Kesenjangan Digitalisasi

**Deskripsi Proyek:**
- **Mata Kuliah:** Big Data Analytics
- **Domain:** Analisis Spasial & Kesenjangan Digital UMKM
- **Cakupan Wilayah:** 6 Kecamatan Surabaya Timur
- **Fokus Kasus:** Sentra Wisata Kuliner (SWK Convention Hall & Deles)

---
## 1. Pendahuluan Analisis BDA & Spasial
Tahap BDA menyatukan hasil ekstraksi dan modeling ke dalam perspektif makro geospasial untuk menjawab kebutuhan pemangku kebijakan:
1. **Agregasi Big Data:** Menghitung indeks adopsi digital per kecamatan di Surabaya Timur.
2. **Pemetaan Spasial:** Visualisasi sebaran UMKM dan klaster sentra kuliner (SWK Convention Hall & SWK Deles).
3. **Gap Analysis:** Mengidentifikasi kecamatan prioritas yang memerlukan intervensi insentif / pendampingan pembayaran digital.


In [ ]:
# 1. Inisialisasi Environment & Koneksi Otomatis Google Drive
import os
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Deteksi apakah kode berjalan di Google Colab
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print('-> Terdeteksi berjalan di Google Colab. Menghubungkan ke Google Drive...')
    from google.colab import drive
    drive.mount('/content/drive')
    
    drive_root = Path('/content/drive/MyDrive')
    target_stage = 'BDA'
    output_dir = None
    
    for p in drive_root.rglob('*'):
        if p.is_dir() and target_stage.lower() == p.name.lower():
            output_dir = p
            break
            
    if not output_dir:
        output_dir = drive_root / 'Kelompok_3_BDA' / target_stage
        output_dir.mkdir(parents=True, exist_ok=True)
        
    OUTPUT_DIR = output_dir
    print(f'-> [SUKSES] Seluruh output BDA akan otomatis tersimpan di Google Drive: {OUTPUT_DIR}')
    
    clean_dir = None
    for p in [Path('/content/data/clean'), Path('data/clean')] + list(drive_root.rglob('clean')):
        if p.is_dir() and (p / 'umkm.parquet').exists():
            clean_dir = p
            break
    if not clean_dir:
        clean_dir = Path('/content/data/clean')
else:
    print('-> Terdeteksi berjalan di environment lokal (VS Code).')
    base_dir = Path.cwd()
    clean_dir = base_dir / 'data' / 'clean' if (base_dir / 'data' / 'clean').exists() else base_dir.parent / 'data' / 'clean'
    OUTPUT_DIR = base_dir / 'reports' / 'bda'
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

df_umkm = pd.read_parquet(clean_dir / 'umkm.parquet')
df_ulasan = pd.read_parquet(clean_dir / 'ulasan.parquet')
print('Data spasial UMKM berhasil dimuat!')


---
## 2. Agregasi Spasial per Kecamatan di Surabaya Timur
Menghitung persentase adopsi pembayaran digital (QRIS, POS, EDC, Merchant Online) per kecamatan.


In [ ]:
# 2.1 Agregasi Sinyal Digital per Kecamatan
bda_agg = df_ulasan.groupby('kecamatan').agg({
    'mention_qris': lambda x: round((x.sum() / len(x)) * 100, 2),
    'mention_pos': lambda x: round((x.sum() / len(x)) * 100, 2),
    'mention_edc': lambda x: round((x.sum() / len(x)) * 100, 2),
    'mention_online': lambda x: round((x.sum() / len(x)) * 100, 2),
    'rating_pengulas': 'mean',
    'ulasan_id': 'count'
}).reset_index()

bda_agg.columns = ['Kecamatan', 'Adopsi QRIS (%)', 'Adopsi POS (%)', 'Adopsi EDC (%)', 'Adopsi Ojol (%)', 'Rata-rata Rating', 'Total Ulasan']
display(bda_agg)


In [ ]:
# 2.2 Visualisasi Perbandingan Adopsi Digital Antar-Kecamatan
plt.figure(figsize=(10, 5))
bar_data = bda_agg.set_index('Kecamatan')[['Adopsi QRIS (%)', 'Adopsi POS (%)', 'Adopsi EDC (%)', 'Adopsi Ojol (%)']]
bar_data.plot(kind='bar', figsize=(11, 5), colormap='viridis')
plt.title('Tingkat Penetrasi Indikator Digitalisasi UMKM Kuliner per Kecamatan di Surabaya Timur', fontsize=13, fontweight='bold')
plt.ylabel('Persentase Penetrasi (%)', fontsize=11)
plt.xlabel('Kecamatan', fontsize=11)
plt.xticks(rotation=15)
plt.legend(title='Indikator')
plt.tight_layout()
save_path = OUTPUT_DIR / '01_penetrasi_digital_kecamatan.png'
plt.savefig(save_path, dpi=300, bbox_inches='tight')
print(f'Grafik tersimpan di: {save_path}')
plt.show()


---
## 3. Analisis Kesenjangan (Gap Analysis) & Rekomendasi Kebijakan
Menganalisis kesenjangan adopsi digital antar-wilayah untuk merumuskan rekomendasi intervensi dinas terkait.


In [ ]:
# 3.1 Perhitungan Indeks Kesenjangan (Digital Gap Index)
bda_agg['Skor Kematangan Rata-Rata'] = (
    bda_agg['Adopsi QRIS (%)'] * 0.35 +
    bda_agg['Adopsi POS (%)'] * 0.25 +
    bda_agg['Adopsi EDC (%)'] * 0.15 +
    bda_agg['Adopsi Ojol (%)'] * 0.25
)

bda_agg['Prioritas Intervensi'] = bda_agg['Skor Kematangan Rata-Rata'].apply(
    lambda s: 'Prioritas Tinggi (Butuh Pendampingan)' if s < 10.0 else (
        'Prioritas Sedang (Perluasan POS/EDC)' if s < 20.0 else 'Mandiri / Maju'
    )
)

display(bda_agg[['Kecamatan', 'Skor Kematangan Rata-Rata', 'Prioritas Intervensi']].sort_values(by='Skor Kematangan Rata-Rata'))


---
## 4. Analisis Khusus Sentra Wisata Kuliner (SWK Convention Hall & Deles)
Mengevaluasi sentra kuliner binaan di Kecamatan Sukolilo sebagai pusat percontohan digitalisasi UMKM.


In [ ]:
# 4.1 Filter Entitas SWK Sukolilo
swk_data = df_umkm[df_umkm['tipe_lokasi'].isin(['SWK', 'SWK_Stan_Survei'])]
print(f'Total entitas terkait Sentra Wisata Kuliner (SWK): {len(swk_data)}')
display(swk_data[['nama_umkm', 'tipe_lokasi', 'alamat']].head(5))


---
## 5. Rekomendasi Aksi untuk Pemangku Kepentingan
Berdasarkan analisis BDA spasial di Surabaya Timur:
1. **Fokus Intervensi QRIS:** Sosialisasi QRIS massal untuk kecamatan perbatasan (Gunung Anyar & Mulyorejo) untuk menggeser UMKM Konvensional menjadi Berkembang.
2. **Penyediaan POS Murah:** Sentra Wisata Kuliner (seperti SWK Convention Hall & Deles) memerlukan standardisasi sistem POS bersama agar pencatatan transaksi lebih akurat dan terintegrasi.
3. **Dukungan Kemitraan Ojol:** Fasilitasi kemitraan kolektif dengan platform online (GoFood/ShopeeFood/GrabFood) agar stan kuliner SWK dapat menjangkau konsumen lebih luas di luar jam makan malam.
